# OAF-Track on SportsMOT: tracking, baselines, evaluation, ablation

Reproduces Tables 3/4, 8 and 9 and Figs. 2–3 of the paper, and writes the SportsMOT **test** submission (Cell 14).

* Runs the detector **once** per sequence and caches detections, Re-ID embeddings and camera-motion (GMC) matrices,
  so every tracker receives *identical* detections and features.
* **OAF-Track**: Kalman filter with GMC, two-stage Hungarian association on a fused IoU + Re-ID cost,
  k-reciprocal (mutual-nearest-neighbour) recovery of lost tracks, observation-centric output.
  The four optional filters evaluated in the paper (ASG-NMS, Mahalanobis gating, occlusion-adaptive weighting,
  tracklet-level temporal NMS) can be switched on individually; `OAF-Track (full, 8 modules)` enables all of them.
* Baselines re-implemented with the association logic of the original papers: IoU-Tracker, SORT, ByteTrack, OC-SORT, BoT-SORT.
* Metrics: HOTA / DetA / AssA / MOTA / IDF1 / MOTP / IDSW / FP / FN / Frag / MT / ML, an exact re-implementation of
  TrackEval's MOTChallenge definitions (no TrackEval install needed). A GT-vs-GT sanity check must give 100 %.

Run the cells top to bottom. **Only Cell 2 (configuration) needs editing.**

In [ ]:
# ── Cell 1: Install dependencies (run once) ─────────────────────────────
import subprocess, sys
pkgs = ['ultralytics', 'opencv-python', 'scipy', 'pandas', 'matplotlib', 'tqdm', 'tabulate']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs, check=False)
print('Done. (torch/torchvision are pulled in by ultralytics if missing)')

In [ ]:
# ── Cell 2: CONFIGURATION  (edit only this cell) ────────────────────────
from pathlib import Path

# Folder that contains the SportsMOT data. Any of these works — the split folder is auto-detected:
#   data/SportsMOT
#   data/SportsMOT/sportsmot_publish/dataset
#   data/SportsMOT/sportsmot_publish/dataset/val
DATA_ROOT = Path(r"data/SportsMOT")
SPLIT     = 'val'            # 'val' has public GT; 'test' has no GT and cannot be evaluated locally

# All outputs (detection cache, tracker txt files, tables, plots) go here
WORK_DIR  = Path(r"outputs/sportsmot")

# Detector (chosen automatically, in this order):
#   1. detector_final.pt  – the frozen, finished fine-tuned model (use this for the paper)
#   2. yolo_runs/.../best.pt – fine-tuned, but may still change if Cell 13 is run
#   3. yolov8s.pt         – COCO model (not fine-tuned)
FINAL_DET      = WORK_DIR / 'detector_final.pt'
FINETUNED      = WORK_DIR / 'yolo_runs' / 'sportsmot_player' / 'weights' / 'best.pt'
DET_WEIGHTS    = str(FINAL_DET) if FINAL_DET.exists() else (str(FINETUNED) if FINETUNED.exists() else 'yolov8s.pt')
IMG_SIZE       = 1280          # SportsMOT frames are 1280x720; 1280 keeps small/far players
DET_CONF_MIN   = 0.05          # keep low-score boxes (ByteTrack-style trackers use them)
YOLO_NMS_IOU   = 0.85          # lenient internal NMS -> dense candidates; our own filters decide
STD_NMS_IOU    = 0.70          # standard NMS applied for baselines (= YOLO default)
# Re-ID appearance model (chosen automatically): your trained SportsReID model if it exists
# (from SportsReID_Train.ipynb), otherwise generic ImageNet ResNet-50.
REID_WEIGHTS   = WORK_DIR / 'reid' / 'sportsreid_r50.pth'
REID_BACKBONE  = 'sportsreid' if REID_WEIGHTS.exists() else 'resnet50'   # or 'resnet18' | 'hsv'
REID_MIN_SCORE = 0.25          # embeddings computed only for boxes above this score (speed)
BATCH          = 8             # frames per GPU batch in detection (use 4 if GPU memory is short)

MAX_SEQS       = None          # None = all sequences; e.g. 3 for a quick test
FORCE_REDETECT = False         # True = recompute the detection cache
PROPOSED_NAME  = 'OAF-Track'   # name of the proposed method in all tables / plots
RESUME         = True          # reuse saved results of Cells 9/10 (checkpoint_*.pkl) instead of re-running

In [ ]:
# ── Cell 3: Imports, paths, dataset discovery ───────────────────────────
import os
# Windows/Anaconda: avoid the OpenMP clash between PyTorch and NumPy/MKL that silently kills the kernel
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '4'); os.environ.setdefault('MKL_NUM_THREADS', '4')
import time, json, math, configparser, warnings
import numpy as np, pandas as pd, cv2
from scipy.optimize import linear_sum_assignment
from scipy.linalg import solve_triangular
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=RuntimeWarning)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def _is_seq_dir(d):
    return d.is_dir() and (d / 'img1').is_dir()

def find_split_dir(root, split):
    root = Path(root)
    if not root.exists(): raise FileNotFoundError(f'DATA_ROOT does not exist: {root}')
    if root.name == split and any(_is_seq_dir(d) for d in root.iterdir()): return root
    for c in [root / split, root / 'dataset' / split, root / 'sportsmot_publish' / 'dataset' / split,
              root / 'dancetrack' / split]:
        if c.is_dir() and any(_is_seq_dir(d) for d in c.iterdir()):
            return c
    for dirpath, dirnames, _ in os.walk(root):          # fallback: search the tree
        p = Path(dirpath)
        if p.name == split and any(_is_seq_dir(p / d) for d in dirnames):
            return p
        dirnames[:] = [d for d in dirnames if d not in ('img1', 'gt')]   # never descend into images
    raise FileNotFoundError(f'No "{split}" folder with sequence sub-folders (containing img1/) under {root}')

def read_seqinfo(seq_dir):
    info = {'frameRate': 25, 'imWidth': None, 'imHeight': None, 'seqLength': None}
    ini = seq_dir / 'seqinfo.ini'
    if ini.exists():
        cp = configparser.ConfigParser(); cp.read(ini)
        s = cp['Sequence']
        info['frameRate'] = int(float(s.get('frameRate', 25)))
        info['imWidth']   = int(s.get('imWidth', 0)) or None
        info['imHeight']  = int(s.get('imHeight', 0)) or None
        info['seqLength'] = int(s.get('seqLength', 0)) or None
    return info

def list_frames(seq_dir):
    files = sorted(p for p in (seq_dir / 'img1').iterdir() if p.suffix.lower() in IMG_EXTS)
    out = []
    for i, p in enumerate(files):
        try:    fno = int(p.stem)
        except ValueError: fno = i + 1
        out.append((fno, p))
    return out

VAL_DIR = find_split_dir(DATA_ROOT, SPLIT)
SEQS = sorted(d for d in VAL_DIR.iterdir() if _is_seq_dir(d))
if MAX_SEQS: SEQS = SEQS[:MAX_SEQS]

# One folder per (detector, Re-ID) combination, so results of different set-ups never mix
TAG         = Path(DET_WEIGHTS).stem + ('' if REID_BACKBONE == 'resnet50' else f'_{REID_BACKBONE}')
BASE_CACHE  = WORK_DIR / 'det_cache' / Path(DET_WEIGHTS).stem      # detections of this detector
CACHE_DIR   = WORK_DIR / 'det_cache' / TAG
TRACKER_DIR = WORK_DIR / 'trackers' / TAG / f'SportsMOT-{SPLIT}'
RESULT_DIR  = WORK_DIR / 'results' / TAG
for d in (CACHE_DIR, TRACKER_DIR, RESULT_DIR): d.mkdir(parents=True, exist_ok=True)

_stem = Path(DET_WEIGHTS).stem
print('DETECTOR  :', DET_WEIGHTS, {'detector_final': '(final fine-tuned model – OK for the paper)',
      'best': '(fine-tuned, NOT the frozen final model)'}.get(_stem, '<-- COCO model, NOT fine-tuned!'))
print('RE-ID     :', REID_BACKBONE, '(trained on SportsMOT train)' if REID_BACKBONE == 'sportsreid'
      else '(generic ImageNet features)')
print('RESULTS   :', RESULT_DIR)
print('Split dir :', VAL_DIR)
print('Sequences :', len(SEQS))
n_gt = 0
for s in SEQS[:5]:
    gt_ok = (s / 'gt' / 'gt.txt').exists(); n_gt += gt_ok
    print(f'  {s.name:28s} frames={len(list_frames(s)):4d}  gt={gt_ok}')
missing_gt = [s.name for s in SEQS if not (s / 'gt' / 'gt.txt').exists()]
assert not missing_gt, f'GT missing for {missing_gt[:5]} … (use the val split)'
print('Work dir  :', WORK_DIR)

In [ ]:
# ── Cell 4: Geometry, Kalman filter, NMS variants ───────────────────────
def iou_matrix(a, b):
    """IoU between xyxy boxes a (N,4) and b (M,4)."""
    a = np.asarray(a, np.float64).reshape(-1, 4); b = np.asarray(b, np.float64).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = aa[:, None] + ab[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)

def xyxy2xyah(b):
    w, h = b[2] - b[0], b[3] - b[1]
    return np.array([b[0] + w / 2, b[1] + h / 2, w / max(h, 1e-6), h], np.float64)

def xyah2xyxy(m):
    w = m[2] * m[3]; h = m[3]
    return np.array([m[0] - w / 2, m[1] - h / 2, m[0] + w / 2, m[1] + h / 2], np.float64)

class KalmanXYAH:
    """Constant-velocity Kalman filter on (cx, cy, aspect, h) — as in DeepSORT/ByteTrack."""
    CHI2_95_4DOF = 9.4877
    def __init__(self):
        self.F = np.eye(8); self.F[:4, 4:] = np.eye(4)
        self.H = np.eye(4, 8)
        self.sp, self.sv = 1 / 20, 1 / 160
    def initiate(self, z):
        h = z[3]
        std = [2*self.sp*h, 2*self.sp*h, 1e-2, 2*self.sp*h, 10*self.sv*h, 10*self.sv*h, 1e-5, 10*self.sv*h]
        return np.r_[z, np.zeros(4)], np.diag(np.square(std))
    def predict(self, mean, cov):
        h = mean[3]
        std = [self.sp*h, self.sp*h, 1e-2, self.sp*h, self.sv*h, self.sv*h, 1e-5, self.sv*h]
        return self.F @ mean, self.F @ cov @ self.F.T + np.diag(np.square(std))
    def project(self, mean, cov):
        h = mean[3]
        R = np.diag(np.square([self.sp*h, self.sp*h, 1e-1, self.sp*h]))
        return self.H @ mean, self.H @ cov @ self.H.T + R
    def update(self, mean, cov, z):
        pm, pc = self.project(mean, cov)
        K = np.linalg.solve(pc, (cov @ self.H.T).T).T
        return mean + K @ (z - pm), cov - K @ pc @ K.T
    def gating_distance(self, mean, cov, zs):
        """Squared Mahalanobis distance between the track state and measurements zs (N,4)."""
        pm, pc = self.project(mean, cov)
        L = np.linalg.cholesky(pc)
        d = solve_triangular(L, (np.asarray(zs) - pm).T, lower=True, check_finite=False)
        return np.sum(d * d, axis=0)

def warp_state(mean, cov, H):
    """Camera-motion compensation: apply 2x3 affine H to the Kalman state (position & velocity)."""
    if H is None: return mean, cov
    R, t = H[:, :2], H[:, 2]
    T = np.eye(8); T[0:2, 0:2] = R; T[4:6, 4:6] = R
    mean = T @ mean; mean[:2] += t
    return mean, T @ cov @ T.T

def nms_standard(boxes, scores, thr):
    order = np.argsort(-scores); keep = []
    sup = np.zeros(len(boxes), bool)
    ious = iou_matrix(boxes, boxes)
    for i in order:
        if sup[i]: continue
        keep.append(i); sup |= ious[i] > thr
    return np.array(keep, int)

def nms_asg(boxes, scores, feats, has_feat, iou_lo=0.55, iou_hi=0.90, sim_thr=0.55, std_thr=0.70):
    """Appearance-Similarity-Guided NMS.
    IoU > iou_hi                 -> always a duplicate (suppress)
    iou_lo < IoU <= iou_hi      -> suppress only if appearance cosine similarity >= sim_thr
                                   (dissimilar = two different, mutually-occluding players -> keep both)
    boxes without an embedding  -> fall back to standard NMS at std_thr"""
    order = np.argsort(-scores); keep = []
    sup = np.zeros(len(boxes), bool)
    ious = iou_matrix(boxes, boxes)
    sims = feats @ feats.T if len(feats) else np.zeros((0, 0))
    for i in order:
        if sup[i]: continue
        keep.append(i)
        both = has_feat[i] & has_feat
        s_app = both & (ious[i] > iou_lo) & (ious[i] <= iou_hi) & (sims[i] >= sim_thr)
        s_std = (~both) & (ious[i] > std_thr)
        sup |= (ious[i] > iou_hi) | s_app | s_std
    return np.array(keep, int)

def linear_assignment(cost, thresh):
    if cost.size == 0:
        return [], list(range(cost.shape[0])), list(range(cost.shape[1]))
    c = np.where(np.isfinite(cost), cost, 1e6)
    r, k = linear_sum_assignment(c)
    matches = [(i, j) for i, j in zip(r, k) if c[i, j] <= thresh]
    mr = {i for i, _ in matches}; mc = {j for _, j in matches}
    return (matches, [i for i in range(cost.shape[0]) if i not in mr],
            [j for j in range(cost.shape[1]) if j not in mc])

print('Utilities ready.')

In [ ]:
# ── Cell 5: Detector + Re-ID encoder + camera-motion (GMC), with caching ──
import gc, logging
logging.getLogger('ultralytics').setLevel(logging.ERROR)   # hide harmless deprecation warnings
# PyTorch / YOLO are imported ONLY if some sequence still needs detection (see bottom of this cell).
# Tracking + evaluation (Cells 6-12) run without PyTorch, which avoids Windows DLL/OpenMP crashes.

class ReIDEncoder:
    """ImageNet CNN appearance embedding (L2-normalised). Falls back to HSV histograms."""
    def __init__(self, kind):
        self.kind = kind
        if kind == 'sportsreid':
            import torchvision
            import torch.nn.functional as TF
            class _SportsReID(torch.nn.Module):          # same architecture as SportsReID_Train.ipynb
                def __init__(self):
                    super().__init__()
                    r = torchvision.models.resnet50(weights=None)
                    r.layer4[0].conv2.stride = (1, 1)
                    r.layer4[0].downsample[0].stride = (1, 1)
                    self.backbone = torch.nn.Sequential(*list(r.children())[:-2])
                    self.bnneck = torch.nn.BatchNorm1d(2048)
                def forward(self, x):
                    f = TF.adaptive_avg_pool2d(self.backbone(x), 1).flatten(1)
                    return TF.normalize(self.bnneck(f), dim=1)
            m = _SportsReID()
            ck = torch.load(REID_WEIGHTS, map_location='cpu', weights_only=False)
            m.load_state_dict(ck['state_dict'], strict=True)
            self.model = m.eval().to(TORCH_DEV); self.dim = 2048
            self.mean = torch.tensor([0.485, 0.456, 0.406], device=TORCH_DEV).view(1, 3, 1, 1)
            self.std  = torch.tensor([0.229, 0.224, 0.225], device=TORCH_DEV).view(1, 3, 1, 1)
            print(f"[ReID] SportsReID loaded (val within-game mAP {ck.get('val_map', float('nan')):.1f})")
        elif kind in ('resnet18', 'resnet50'):
            try:
                import torchvision
                W = {'resnet18': torchvision.models.ResNet18_Weights.IMAGENET1K_V1,
                     'resnet50': torchvision.models.ResNet50_Weights.IMAGENET1K_V2}[kind]
                m = getattr(torchvision.models, kind)(weights=W)
                m.fc = torch.nn.Identity()
                self.model = m.eval().to(TORCH_DEV)
                self.dim = 512 if kind == 'resnet18' else 2048
                self.mean = torch.tensor([0.485, 0.456, 0.406], device=TORCH_DEV).view(1, 3, 1, 1)
                self.std  = torch.tensor([0.229, 0.224, 0.225], device=TORCH_DEV).view(1, 3, 1, 1)
            except Exception as e:
                print(f'[ReID] could not load {kind} ({e}); falling back to HSV histograms'); self.kind = 'hsv'
        if self.kind == 'hsv':
            self.dim = 2 * 16 * 8
    def _crops(self, img, boxes, size):
        H, W = img.shape[:2]; out = []
        for b in boxes:
            x1, y1 = int(max(0, b[0])), int(max(0, b[1])); x2, y2 = int(min(W, b[2])), int(min(H, b[3]))
            if x2 - x1 < 2 or y2 - y1 < 2: out.append(np.zeros((size[1], size[0], 3), np.uint8)); continue
            out.append(cv2.resize(img[y1:y2, x1:x2], size))
        return out
    def __call__(self, img, boxes):
        if len(boxes) == 0: return np.zeros((0, self.dim), np.float32)
        if self.kind == 'hsv':
            feats = []
            for c in self._crops(img, boxes, (64, 128)):
                hsv = cv2.cvtColor(c, cv2.COLOR_BGR2HSV); hs = []
                for part in (hsv[16:72], hsv[72:120]):          # torso / legs
                    h = cv2.calcHist([part], [0, 1], None, [16, 8], [0, 180, 0, 256]).ravel()
                    hs.append(h / (h.sum() + 1e-6))
                feats.append(np.sqrt(np.concatenate(hs)))
            f = np.array(feats, np.float32)
        else:
            crops = np.stack(self._crops(img, boxes, (128, 256)))[..., ::-1].copy()
            with torch.no_grad():
                x = torch.from_numpy(crops).to(TORCH_DEV).permute(0, 3, 1, 2).float() / 255.
                x = (x - self.mean) / self.std
                with torch.autocast(device_type='cuda', enabled=(TORCH_DEV == 'cuda')):
                    f = self.model(x).float().cpu().numpy()
        return f / (np.linalg.norm(f, axis=1, keepdims=True) + 1e-12)

class SparseFlowGMC:
    """Global motion (camera pan/zoom) estimate between consecutive frames — as in BoT-SORT."""
    def __init__(self, downscale=2):
        self.ds = downscale; self.prev = None; self.prev_pts = None
    def apply(self, frame):
        H = np.eye(2, 3)
        g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        g = cv2.resize(g, (g.shape[1] // self.ds, g.shape[0] // self.ds))
        pts = cv2.goodFeaturesToTrack(g, maxCorners=1000, qualityLevel=0.01, minDistance=1, blockSize=3)
        if self.prev is not None and self.prev_pts is not None and len(self.prev_pts) > 10:
            nxt, st, _ = cv2.calcOpticalFlowPyrLK(self.prev, g, self.prev_pts, None)
            if nxt is not None:
                ok = st.ravel() == 1
                if ok.sum() > 10:
                    A, _ = cv2.estimateAffinePartial2D(self.prev_pts[ok], nxt[ok], method=cv2.RANSAC)
                    if A is not None:
                        H = A.copy(); H[:, 2] *= self.ds
        self.prev, self.prev_pts = g, pts
        return H

def detect_sequence(seq_dir, cache_file, detector, encoder, batch=globals().get('BATCH', 8)):
    frames = list_frames(seq_dir)
    F, B, S, E, WARP = [], [], [], [], []
    gmc = SparseFlowGMC(); t_det = t_reid = t_gmc = 0.0
    for k in tqdm(range(0, len(frames), batch), desc=f'detect {seq_dir.name}', leave=False):
        chunk = frames[k:k + batch]
        imgs = [cv2.imread(str(p)) for _, p in chunk]
        t0 = time.perf_counter()
        res = detector.predict(imgs, imgsz=IMG_SIZE, conf=DET_CONF_MIN, iou=YOLO_NMS_IOU, classes=[0],
                               max_det=300, verbose=False, device=DEVICE, half=(TORCH_DEV == 'cuda'))
        t_det += time.perf_counter() - t0
        for (fno, _), img, r in zip(chunk, imgs, res):
            b = r.boxes.xyxy.cpu().numpy().astype(np.float32); s = r.boxes.conf.cpu().numpy().astype(np.float32)
            t1 = time.perf_counter()
            e = np.zeros((len(b), encoder.dim), np.float16); m = s >= REID_MIN_SCORE
            if m.any(): e[m] = encoder(img, b[m]).astype(np.float16)
            t2 = time.perf_counter()
            WARP.append(gmc.apply(img)); t3 = time.perf_counter()
            t_reid += t2 - t1; t_gmc += t3 - t2
            F.append(np.full(len(b), fno, np.int32)); B.append(b); S.append(s); E.append(e)
    np.savez_compressed(cache_file,
        frame=np.concatenate(F), boxes=np.concatenate(B).reshape(-1, 4), scores=np.concatenate(S),
        feats=np.concatenate(E).reshape(-1, encoder.dim), frame_ids=np.array([f for f, _ in frames], np.int32),
        warps=np.stack(WARP), timing=np.array([t_det, t_reid, t_gmc, len(frames)]))

def reembed_sequence(seq_dir, src_file, cache_file, encoder):
    """Re-use the cached YOLO boxes + camera motion of src_file; recompute only the Re-ID features."""
    z = np.load(src_file)
    fr, bx, sc = z['frame'], z['boxes'], z['scores']
    E = np.zeros((len(bx), encoder.dim), np.float16)
    paths = dict(list_frames(seq_dir)); t_reid = 0.0
    for f in tqdm(np.unique(fr), desc=f're-embed {seq_dir.name}', leave=False):
        ix = np.where((fr == f) & (sc >= REID_MIN_SCORE))[0]
        if not len(ix):
            continue
        img = cv2.imread(str(paths[int(f)]))
        t0 = time.perf_counter()
        E[ix] = encoder(img, bx[ix]).astype(np.float16)
        t_reid += time.perf_counter() - t0
    tim = z['timing'].copy(); tim[1] = t_reid
    np.savez_compressed(cache_file, frame=fr, boxes=bx, scores=sc, feats=E,
                        frame_ids=z['frame_ids'], warps=z['warps'], timing=tim)

def ensure_cache(seqs, cache_dir, base_dir, label):
    """Detect (or only re-embed, if this detector's boxes are already cached) every missing sequence."""
    global DEVICE, TORCH_DEV, torch
    cache_dir.mkdir(parents=True, exist_ok=True)
    todo = [s for s in seqs if FORCE_REDETECT or not (cache_dir / f'{s.name}.npz').exists()]
    if not todo:
        print(f'All {label} sequences already cached – PyTorch not loaded (safe mode for tracking).')
        return False
    import torch
    DEVICE = 0 if torch.cuda.is_available() else 'cpu'
    TORCH_DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Device:', TORCH_DEV, '|', torch.cuda.get_device_name(0) if TORCH_DEV == 'cuda' else 'NO GPU – slow')
    encoder = ReIDEncoder(REID_BACKBONE)
    reuse = [s for s in todo if cache_dir != base_dir and not FORCE_REDETECT
             and (base_dir / f'{s.name}.npz').exists()]
    full = [s for s in todo if s not in reuse]
    if reuse:
        print(f'Re-embedding {len(reuse)} {label} sequences with {encoder.kind} (YOLO boxes re-used) …')
        for s in tqdm(reuse, desc=f're-embed {label}'):
            reembed_sequence(s, base_dir / f'{s.name}.npz', cache_dir / f'{s.name}.npz', encoder)
    if full:
        from ultralytics import YOLO
        logging.getLogger('ultralytics').setLevel(logging.ERROR)
        detector = YOLO(DET_WEIGHTS)
        print(f'Detecting {len(full)} {label} sequences with {DET_WEIGHTS} (Re-ID: {encoder.kind}) …')
        for s in tqdm(full, desc=f'detect {label}'):
            detect_sequence(s, cache_dir / f'{s.name}.npz', detector, encoder)
        del detector
    del encoder; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return True

class SeqCache:
    """Per-frame access to ONE sequence's cached detections.
    * boxes with score < 0.1 are dropped at load time (no tracker uses them; greedy NMS results are unchanged)
    * embeddings are mean-centred per sequence and re-normalised, so cosine thresholds are meaningful"""
    MIN_SCORE = 0.1
    def __init__(self, path):
        z = np.load(path)
        self.frame_ids = z['frame_ids']; self.timing = z['timing']
        sc = z['scores']; keep = sc >= self.MIN_SCORE
        fr, bx, sc = z['frame'][keep], z['boxes'][keep].astype(np.float64), sc[keep].astype(np.float64)
        ft = z['feats'][keep].astype(np.float32)
        has = np.linalg.norm(ft, axis=1) > 0
        if has.sum() > 1:
            mu = ft[has].mean(0); ft[has] -= mu
            ft[has] /= np.linalg.norm(ft[has], axis=1, keepdims=True) + 1e-12
        self.warps = dict(zip(self.frame_ids.tolist(), z['warps']))
        order = np.argsort(fr, kind='stable')
        fr, bx, sc, ft, has = fr[order], bx[order], sc[order], ft[order], has[order]
        starts = np.searchsorted(fr, self.frame_ids, 'left'); ends = np.searchsorted(fr, self.frame_ids, 'right')
        self.idx = {int(f): (a, b) for f, a, b in zip(self.frame_ids, starts, ends)}
        self.bx, self.sc, self.ft, self.has = bx, sc, ft, has
    def get(self, f):
        a, b = self.idx[int(f)]
        return self.bx[a:b], self.sc[a:b], self.ft[a:b], self.has[a:b]

class LazyCaches:
    """Loads a sequence only when it is needed and keeps just ONE in RAM (avoids kernel crashes)."""
    def __init__(self, paths):
        self.paths = paths; self._name = None; self._obj = None
    def __getitem__(self, name):
        if name != self._name:
            self._obj = None; gc.collect()
            self._obj = SeqCache(self.paths[name]); self._name = name
        return self._obj
    def __len__(self): return len(self.paths)

if ensure_cache(SEQS, CACHE_DIR, BASE_CACHE, 'val'):
    print('Done. TIP: restart the kernel and run all cells again – tracking then runs without PyTorch loaded.')
CACHES = LazyCaches({s.name: CACHE_DIR / f'{s.name}.npz' for s in SEQS})

tim = np.sum([np.load(CACHES.paths[s.name])['timing'] for s in SEQS], axis=0)
DET_FPS = tim[3] / max(tim[0], 1e-9); REID_FPS = tim[3] / max(tim[1], 1e-9); GMC_FPS = tim[3] / max(tim[2], 1e-9)
print(f'Cached {len(CACHES)} sequences | detector {DET_FPS:.1f} FPS | Re-ID {REID_FPS:.1f} FPS | GMC {GMC_FPS:.1f} FPS')
c0 = CACHES[SEQS[0].name]; b, s, _, _ = c0.get(c0.frame_ids[0])
print(f'Smoke test {SEQS[0].name} frame {c0.frame_ids[0]}: {len(b)} boxes (score>=0.1), {(s >= 0.5).sum()} with score>=0.5')
assert len(c0.sc) > 0, 'Detector produced no boxes – check DET_WEIGHTS / images'

In [ ]:
# ── Cell 6: Baseline trackers (IoU-Tracker, SORT, OC-SORT, ByteTrack, BoT-SORT) ──
KF = KalmanXYAH()

# ---------- IoU-Tracker (Bochinski et al., 2017; online variant) ----------
class IoUTracker:
    frame_filter = 'nms'; needs_warp = False
    def __init__(self, frame_rate=25, sigma_l=0.3, sigma_h=0.5, sigma_iou=0.5, t_min=2):
        self.sl, self.sh, self.si, self.tmin = sigma_l, sigma_h, sigma_iou, t_min
        self.tracks = []; self.nid = 0
    def update(self, boxes, scores, feats=None, has=None, warp=None):
        m = scores >= self.sl; boxes, scores = boxes[m], scores[m]
        used = np.zeros(len(boxes), bool); alive = []
        if len(self.tracks) and len(boxes):
            ious = iou_matrix(np.array([t['box'] for t in self.tracks]), boxes)
        for ti, t in enumerate(self.tracks):
            if len(boxes):
                cand = np.where(~used, ious[ti], -1); j = int(np.argmax(cand))
                if cand[j] >= self.si:
                    used[j] = True; t['box'] = boxes[j]; t['len'] += 1; t['max'] = max(t['max'], scores[j])
                    alive.append(t); continue
        for j in np.where(~used)[0]:
            self.nid += 1; alive.append({'id': self.nid, 'box': boxes[j], 'len': 1, 'max': scores[j]})
        self.tracks = alive
        return [(t['id'], t['box'], t['max']) for t in alive if t['len'] >= self.tmin and t['max'] >= self.sh]

# ---------- SORT / OC-SORT (shared implementation, OC components switchable) ----------
def speed_direction(b1, b2):
    c1 = ((b1[0] + b1[2]) / 2, (b1[1] + b1[3]) / 2); c2 = ((b2[0] + b2[2]) / 2, (b2[1] + b2[3]) / 2)
    v = np.array([c2[1] - c1[1], c2[0] - c1[0]]); return v / (np.linalg.norm(v) + 1e-6)

class OCTrack:
    def __init__(self, box, score, tid, delta_t):
        self.mean, self.cov = KF.initiate(xyxy2xyah(box)); self.id = tid; self.score = score
        self.tsu = 0; self.hits = 1; self.hit_streak = 1; self.age = 0; self.dt = delta_t
        self.last_obs = np.asarray(box, float); self.obs = {0: self.last_obs}
        self.vel = np.zeros(2); self.snap = (self.mean.copy(), self.cov.copy())
    def k_prev_obs(self):
        for i in range(self.dt):
            d = self.dt - i
            if self.age - d in self.obs: return self.obs[self.age - d]
        return self.obs[max(self.obs)]
    def predict(self):
        if self.mean[7] + self.mean[3] <= 0: self.mean[7] = 0
        self.mean, self.cov = KF.predict(self.mean, self.cov)
        self.age += 1
        if self.tsu > 0: self.hit_streak = 0
        self.tsu += 1
        return xyah2xyxy(self.mean)
    def update(self, box, score, oru):
        box = np.asarray(box, float)
        prev = self.k_prev_obs(); self.vel = speed_direction(prev, box)
        gap = self.tsu
        if oru and gap > 1:                              # Observation-centric Re-Update (ORU)
            self.mean, self.cov = self.snap
            for i in range(1, gap + 1):
                virt = self.last_obs + (box - self.last_obs) * i / gap
                self.mean, self.cov = KF.predict(self.mean, self.cov)
                self.mean, self.cov = KF.update(self.mean, self.cov, xyxy2xyah(virt))
        else:
            self.mean, self.cov = KF.update(self.mean, self.cov, xyxy2xyah(box))
        self.snap = (self.mean.copy(), self.cov.copy())
        self.last_obs = box; self.obs[self.age] = box; self.score = score
        self.tsu = 0; self.hits += 1; self.hit_streak += 1

class OCSORT:
    """OC-SORT (Cao et al., CVPR 2023): OCM (velocity-direction consistency), OCR (re-association
    with last observation) and ORU (virtual-trajectory re-update). With all three off -> SORT."""
    frame_filter = 'nms'; needs_warp = False
    def __init__(self, frame_rate=25, det_thresh=0.6, max_age=30, min_hits=3, iou_thr=0.3, delta_t=3,
                 inertia=0.2, ocm=True, ocr=True, oru=True, output_obs=True):
        self.__dict__.update(dict(det_thresh=det_thresh, max_age=max_age, min_hits=min_hits, iou_thr=iou_thr,
                                  delta_t=delta_t, inertia=inertia, ocm=ocm, ocr=ocr, oru=oru, output_obs=output_obs))
        self.tracks = []; self.frame = 0; self.nid = 0
    def update(self, boxes, scores, feats=None, has=None, warp=None):
        self.frame += 1
        m = scores > self.det_thresh; dets, dsc = boxes[m], scores[m]
        preds = np.array([t.predict() for t in self.tracks]).reshape(-1, 4)
        ok = np.all(np.isfinite(preds), 1)
        self.tracks = [t for t, o in zip(self.tracks, ok) if o]; preds = preds[ok]
        ious = iou_matrix(dets, preds)                                   # (D, T)
        cost = -ious
        if self.ocm and len(self.tracks) and len(dets):
            prev = np.array([t.k_prev_obs() for t in self.tracks])
            vel = np.array([t.vel for t in self.tracks])                 # (T, 2) = (dy, dx)
            pc = np.stack([(prev[:, 0] + prev[:, 2]) / 2, (prev[:, 1] + prev[:, 3]) / 2], 1)
            dc = np.stack([(dets[:, 0] + dets[:, 2]) / 2, (dets[:, 1] + dets[:, 3]) / 2], 1)
            dx = dc[:, None, 0] - pc[None, :, 0]; dy = dc[:, None, 1] - pc[None, :, 1]
            n = np.sqrt(dx ** 2 + dy ** 2) + 1e-6; dx, dy = dx / n, dy / n
            cos = np.clip(vel[None, :, 1] * dx + vel[None, :, 0] * dy, -1, 1)
            ang = (np.pi / 2 - np.abs(np.arccos(cos))) / np.pi
            valid = np.array([len(t.obs) > 1 for t in self.tracks], float)
            cost = -(ious + ang * valid[None, :] * self.inertia * dsc[:, None])
        cost = np.where(ious < self.iou_thr, 1e6, cost)
        matches, ud, ut = linear_assignment(cost, 1e5)
        for d, t in matches: self.tracks[t].update(dets[d], dsc[d], self.oru)
        if self.ocr and ud and ut:                                       # Observation-Centric Recovery
            lo = np.array([self.tracks[t].last_obs for t in ut])
            i2 = iou_matrix(dets[ud], lo)
            m2, _, _ = linear_assignment(np.where(i2 < self.iou_thr, 1e6, -i2), 1e5)
            done_d, done_t = set(), set()
            for a, b in m2:
                self.tracks[ut[b]].update(dets[ud[a]], dsc[ud[a]], self.oru); done_d.add(ud[a]); done_t.add(ut[b])
            ud = [d for d in ud if d not in done_d]
        for d in ud:
            self.nid += 1; self.tracks.append(OCTrack(dets[d], dsc[d], self.nid, self.delta_t))
        out = []
        for t in self.tracks:
            if t.tsu < 1 and (t.hit_streak >= self.min_hits or self.frame <= self.min_hits):
                out.append((t.id, t.last_obs if self.output_obs else xyah2xyxy(t.mean), t.score))
        self.tracks = [t for t in self.tracks if t.tsu <= self.max_age]
        return out

def SORT(frame_rate=25):
    """SORT (Bewley et al., 2016) with its default hyper-parameters."""
    return OCSORT(frame_rate, det_thresh=0.5, max_age=1, min_hits=3, iou_thr=0.3,
                  ocm=False, ocr=False, oru=False, output_obs=False)

# ---------- ByteTrack / BoT-SORT ----------
class STrack:
    def __init__(self, box, score, feat=None):
        self.box = np.asarray(box, float); self.score = float(score)
        self.mean = self.cov = None; self.id = 0; self.state = 'new'; self.is_activated = False
        self.frame_id = self.start_frame = 0; self.tracklet_len = 0
        self.feat = None; self.curr_feat = feat
    @property
    def tlbr(self):
        return self.box if self.mean is None else xyah2xyxy(self.mean[:4])
    @property
    def end_frame(self): return self.frame_id
    def update_feat(self, f, alpha=0.9):
        if f is None: return
        self.feat = f.copy() if self.feat is None else alpha * self.feat + (1 - alpha) * f
        self.feat /= np.linalg.norm(self.feat) + 1e-12
    def predict(self):
        m = self.mean.copy()
        if self.state != 'tracked': m[7] = 0
        self.mean, self.cov = KF.predict(m, self.cov)
    def activate(self, frame_id, tid):
        self.id = tid; self.mean, self.cov = KF.initiate(xyxy2xyah(self.box))
        self.update_feat(self.curr_feat)
        self.state = 'tracked'; self.is_activated = frame_id == 1
        self.frame_id = self.start_frame = frame_id; self.tracklet_len = 0
        self.last_box = self.box.copy()
    def update(self, det, frame_id, feat_update=True, reactivate=False):
        self.mean, self.cov = KF.update(self.mean, self.cov, xyxy2xyah(det.box))
        if feat_update: self.update_feat(det.curr_feat)
        self.state = 'tracked'; self.is_activated = True; self.score = det.score
        self.frame_id = frame_id; self.tracklet_len = 0 if reactivate else self.tracklet_len + 1
        self.last_box = det.box

def iou_dist(tracks, dets):
    if not tracks or not dets: return np.zeros((len(tracks), len(dets)))
    return 1 - iou_matrix(np.array([t.tlbr for t in tracks]), np.array([d.box for d in dets]))

def fuse_score(cost, dets):
    if cost.size == 0: return cost
    return 1 - (1 - cost) * np.array([d.score for d in dets])[None, :]

def emb_dist(tracks, dets):
    """Cosine distance in [0, 2]; 2 when either side has no embedding."""
    D = np.full((len(tracks), len(dets)), 2.0)
    ti = [i for i, t in enumerate(tracks) if t.feat is not None]
    di = [j for j, d in enumerate(dets) if d.curr_feat is not None]
    if ti and di:
        T = np.array([tracks[i].feat for i in ti]); X = np.array([dets[j].curr_feat for j in di])
        D[np.ix_(ti, di)] = np.clip(1 - T @ X.T, 0, 2)
    return D

def joint(a, b):
    seen = {t.id for t in a}; return a + [t for t in b if t.id not in seen]

def sub(a, b):
    ids = {t.id for t in b}; return [t for t in a if t.id not in ids]

def remove_duplicates(tracked, lost):
    if not tracked or not lost: return tracked, lost
    d = iou_dist(tracked, lost); dup_a, dup_b = set(), set()
    for p, q in zip(*np.where(d < 0.15)):
        ta = tracked[p].frame_id - tracked[p].start_frame; tb = lost[q].frame_id - lost[q].start_frame
        (dup_b if ta > tb else dup_a).add(q if ta > tb else p)
    return [t for i, t in enumerate(tracked) if i not in dup_a], [t for i, t in enumerate(lost) if i not in dup_b]

class ByteTrack:
    """ByteTrack (Zhang et al., ECCV 2022). with_reid + with_gmc -> BoT-SORT (Aharon et al., 2022)."""
    frame_filter = 'nms'
    def __init__(self, frame_rate=25, track_thresh=0.6, low_thresh=0.1, match_thresh=0.8, new_track_thresh=0.7,
                 buffer=30, with_reid=False, with_gmc=False, proximity_thresh=0.5, appearance_thresh=0.25):
        self.__dict__.update(dict(track_thresh=track_thresh, low_thresh=low_thresh, match_thresh=match_thresh,
            new_track_thresh=new_track_thresh, with_reid=with_reid, with_gmc=with_gmc,
            proximity_thresh=proximity_thresh, appearance_thresh=appearance_thresh))
        self.needs_warp = with_gmc
        self.max_lost = int(frame_rate / 30.0 * buffer)
        self.tracked, self.lost = [], []; self.frame = 0; self.nid = 0
    def _new_id(self):
        self.nid += 1; return self.nid
    def _first_cost(self, tracks, dets):
        d = iou_dist(tracks, dets)
        if not self.with_reid: return fuse_score(d, dets)
        far = d > self.proximity_thresh
        d = fuse_score(d, dets)
        e = emb_dist(tracks, dets) / 2.0
        e[e > self.appearance_thresh] = 1.0; e[far] = 1.0
        return np.minimum(d, e)
    def update(self, boxes, scores, feats=None, has=None, warp=None):
        self.frame += 1; fr = self.frame
        use_f = self.with_reid and feats is not None
        mk = lambda i: STrack(boxes[i], scores[i], feats[i] if (use_f and has[i]) else None)
        high = [mk(i) for i in np.where(scores >= self.track_thresh)[0]]
        low = [mk(i) for i in np.where((scores > self.low_thresh) & (scores < self.track_thresh))[0]]
        unconf = [t for t in self.tracked if not t.is_activated]
        tracked = [t for t in self.tracked if t.is_activated]
        pool = joint(tracked, self.lost)
        for t in pool: t.predict()
        if self.with_gmc and warp is not None:
            for t in pool + unconf: t.mean, t.cov = warp_state(t.mean, t.cov, warp)
        act, ref, lost, rem = [], [], [], []
        m, ut, ud = linear_assignment(self._first_cost(pool, high), self.match_thresh)
        for i, j in m:
            t = pool[i]
            if t.state == 'tracked': t.update(high[j], fr); act.append(t)
            else: t.update(high[j], fr, reactivate=True); ref.append(t)
        rt = [pool[i] for i in ut if pool[i].state == 'tracked']
        m, ut2, _ = linear_assignment(iou_dist(rt, low), 0.5)
        for i, j in m:
            rt[i].update(low[j], fr, feat_update=False); act.append(rt[i])
        for i in ut2:
            t = rt[i]
            if t.state != 'lost': t.state = 'lost'; lost.append(t)
        rh = [high[j] for j in ud]
        m, uu, ud = linear_assignment(self._first_cost(unconf, rh), 0.7)
        for i, j in m: unconf[i].update(rh[j], fr); act.append(unconf[i])
        for i in uu: unconf[i].state = 'removed'; rem.append(unconf[i])
        for j in ud:
            if rh[j].score >= self.new_track_thresh: rh[j].activate(fr, self._new_id()); act.append(rh[j])
        for t in self.lost:
            if fr - t.end_frame > self.max_lost: t.state = 'removed'; rem.append(t)
        self.tracked = joint(joint([t for t in self.tracked if t.state == 'tracked'], act), ref)
        self.lost = sub(self.lost, self.tracked) + lost
        self.lost = sub(self.lost, rem)
        self.tracked, self.lost = remove_duplicates(self.tracked, self.lost)
        return [(t.id, t.tlbr, t.score) for t in self.tracked if t.is_activated]

def BoTSORT(frame_rate=25):
    return ByteTrack(frame_rate, with_reid=True, with_gmc=True)

print('Baselines ready.')

In [ ]:
# ── Cell 7: Proposed tracker — OAF-Track (Occlusion-aware Appearance-Filtering) ──
class OAFTrack(ByteTrack):
    """Proposed pipeline (each block can be switched off for the ablation study):
      [asg_nms]   frame-level Appearance-Similarity-Guided NMS instead of plain IoU-NMS
      [gmc]       camera-motion compensation of the Kalman states
      [maha]      Kalman + Mahalanobis (chi^2, 95 %) gating of every association
      [reid]      Hungarian assignment on a fused (IoU, cosine Re-ID) cost
      [adaptive]  occlusion-aware weighting: the Re-ID weight shrinks as a track becomes occluded, and
                  the appearance memory is only updated from clean (unoccluded, confident) detections
      [recovery]  k-reciprocal (mutual nearest-neighbour) Re-ID recovery of lost tracks
      [tnms]      tracklet-level temporal NMS: two live tracks overlapping for >= 3 frames -> younger one removed
      [obs_out]   observation-centric output: report the matched detection box instead of the Kalman-smoothed
                  box (the KF lags fast-moving players; this is what gives OC-SORT its high DetA/MOTP)
    Stage 2 (low-score detections, IoU only) follows ByteTrack."""
    def __init__(self, frame_rate=25, asg_nms=True, gmc=True, maha=True, reid=True, adaptive=True,
                 recovery=True, tnms=True, obs_out=True, w_app=0.5, app_gate=0.35, recov_thr=0.25,
                 tnms_iou=0.8, tnms_len=3):
        super().__init__(frame_rate, with_reid=reid, with_gmc=gmc)
        self.obs_out = obs_out
        self.frame_filter = 'asg' if asg_nms else 'nms'
        self.maha, self.adaptive, self.recovery, self.tnms = maha, adaptive, recovery and reid, tnms
        self.w_app, self.app_gate, self.recov_thr = w_app, app_gate, recov_thr
        self.tnms_iou, self.tnms_len = tnms_iou, tnms_len
        self.overlap = {}

    # ---- helpers -------------------------------------------------------------
    def _gate(self, tracks, dets, iou_d):
        """Boolean matrix of admissible pairs."""
        if not self.maha: return iou_d < 0.9                         # plain IoU gate (IoU > 0.1)
        g = np.zeros_like(iou_d, bool)
        if len(tracks) and len(dets):
            Z = np.array([xyxy2xyah(d.box) for d in dets])
            for i, t in enumerate(tracks):
                g[i] = KF.gating_distance(t.mean, t.cov, Z) <= KalmanXYAH.CHI2_95_4DOF
        return g | (iou_d < 0.5)                                     # strong overlap always admissible

    def _occlusion(self, tracks):
        if len(tracks) < 2: return np.zeros(len(tracks))
        B = np.array([t.tlbr for t in tracks]); I = iou_matrix(B, B); np.fill_diagonal(I, 0)
        return I.max(1)

    def _fused_cost(self, tracks, dets):
        iou_d = iou_dist(tracks, dets)
        if iou_d.size == 0: return iou_d
        cost = fuse_score(iou_d, dets)
        if self.with_reid:
            app = emb_dist(tracks, dets) / 2.0                       # [0, 1]
            w = np.full(len(tracks), self.w_app)
            if self.adaptive: w = w * (1 - self._occlusion(tracks))
            w = w[:, None] * (app < 1.0)                            # no embedding -> motion only
            cost = (1 - w) * cost + w * app
            cost[(app > self.app_gate) & (iou_d > 0.5)] = np.inf     # dissimilar AND far apart
        cost[~self._gate(tracks, dets, iou_d)] = np.inf
        return cost

    def _clean(self, dets):
        """Detections whose appearance is reliable enough to update the track's feature memory."""
        if not self.adaptive or not dets: return [True] * len(dets)
        B = np.array([d.box for d in dets]); I = iou_matrix(B, B); np.fill_diagonal(I, 0)
        mx = I.max(1) if len(dets) > 1 else np.zeros(1)
        return [(d.score >= self.track_thresh) and (o < 0.3) for d, o in zip(dets, mx)]

    # ---- main update ---------------------------------------------------------
    def update(self, boxes, scores, feats=None, has=None, warp=None):
        self.frame += 1; fr = self.frame
        mk = lambda i: STrack(boxes[i], scores[i], feats[i] if (self.with_reid and has[i]) else None)
        high = [mk(i) for i in np.where(scores >= self.track_thresh)[0]]
        low = [mk(i) for i in np.where((scores > self.low_thresh) & (scores < self.track_thresh))[0]]
        clean = dict(zip(map(id, high), self._clean(high)))
        unconf = [t for t in self.tracked if not t.is_activated]
        tracked = [t for t in self.tracked if t.is_activated]
        pool = joint(tracked, self.lost)
        for t in pool: t.predict()
        if self.with_gmc and warp is not None:
            for t in pool + unconf: t.mean, t.cov = warp_state(t.mean, t.cov, warp)
        act, ref, lost, rem = [], [], [], []

        def assign(t, d, reactivate=False):
            t.update(d, fr, feat_update=clean.get(id(d), False), reactivate=reactivate)
            (ref if reactivate else act).append(t)

        # Stage 1: high-score detections, fused cost + gating
        m, ut, ud = linear_assignment(self._fused_cost(pool, high), self.match_thresh)
        for i, j in m: assign(pool[i], high[j], reactivate=pool[i].state != 'tracked')
        # Stage 2: low-score detections, IoU (+ Mahalanobis gate)
        rt = [pool[i] for i in ut if pool[i].state == 'tracked']
        c2 = iou_dist(rt, low)
        if c2.size: c2[~self._gate(rt, low, c2)] = np.inf
        m, ut2, _ = linear_assignment(c2, 0.5)
        for i, j in m: assign(rt[i], low[j])
        left = [rt[i] for i in ut2] + [pool[i] for i in ut if pool[i].state != 'tracked']
        rh = [high[j] for j in ud]
        # Stage 3: k-reciprocal Re-ID recovery (lost / just-missed tracks vs. remaining high dets)
        if self.recovery and left and rh:
            A = emb_dist(left, rh) / 2.0
            nn_t = A.argmin(1); nn_d = A.argmin(0); used = set()
            for i, t in enumerate(left):
                j = nn_t[i]
                if nn_d[j] != i or A[i, j] > self.recov_thr: continue          # not mutual NN / too far
                gap = fr - t.end_frame
                pc = t.tlbr; dc = rh[j].box; h = max(pc[3] - pc[1], 1.0)
                dist = np.hypot((pc[0] + pc[2] - dc[0] - dc[2]) / 2, (pc[1] + pc[3] - dc[1] - dc[3]) / 2)
                if dist > h * (1.0 + 0.3 * gap): continue                      # spatially implausible
                assign(t, rh[j], reactivate=t.state != 'tracked'); used.add(j)
            left = [t for t in left if t.frame_id != fr]
            rh = [d for k, d in enumerate(rh) if k not in used]
        for t in left:
            if t.state == 'tracked': t.state = 'lost'; lost.append(t)
        # Unconfirmed tracks
        m, uu, ud = linear_assignment(self._fused_cost(unconf, rh), 0.7)
        for i, j in m: assign(unconf[i], rh[j])
        for i in uu: unconf[i].state = 'removed'; rem.append(unconf[i])
        for j in ud:
            if rh[j].score >= self.new_track_thresh: rh[j].activate(fr, self._new_id()); act.append(rh[j])
        for t in self.lost:
            if fr - t.end_frame > self.max_lost: t.state = 'removed'; rem.append(t)
        self.tracked = joint(joint([t for t in self.tracked if t.state == 'tracked'], act), ref)
        self.lost = sub(sub(self.lost, self.tracked) + lost, rem)
        self.tracked, self.lost = remove_duplicates(self.tracked, self.lost)
        # Tracklet-level temporal NMS
        if self.tnms:
            live = [t for t in self.tracked if t.is_activated and t.frame_id == fr]
            new_ov = {}
            if len(live) > 1:
                I = iou_matrix(np.array([t.tlbr for t in live]), np.array([t.tlbr for t in live]))
                for a, b in zip(*np.where(np.triu(I, 1) > self.tnms_iou)):
                    key = (min(live[a].id, live[b].id), max(live[a].id, live[b].id))
                    new_ov[key] = self.overlap.get(key, 0) + 1
            self.overlap = new_ov; kill = set()
            for (a, b), n in new_ov.items():
                if n >= self.tnms_len:
                    ta = next(t for t in live if t.id == a); tb = next(t for t in live if t.id == b)
                    kill.add(ta.id if ta.start_frame > tb.start_frame else tb.id)
            if kill:
                for t in self.tracked:
                    if t.id in kill: t.state = 'removed'
                self.tracked = [t for t in self.tracked if t.id not in kill]
        box = (lambda t: t.last_box) if self.obs_out else (lambda t: t.tlbr)
        return [(t.id, box(t), t.score) for t in self.tracked if t.is_activated and t.frame_id == fr]

P = PROPOSED_NAME
# Paper configuration ("OAF-Track"): the four filters that did not help in the leave-one-out ablation on
# BOTH datasets (ASG-NMS, Mahalanobis gating, occlusion-adaptive weighting, temporal NMS) are switched off.
# "(full, 8 modules)" keeps every module and is the starting point of the ablation (Section 5.5 of the paper).
TRACKERS = {                                   # name -> factory(frame_rate)
    P:                   lambda fr: OAFTrack(fr, asg_nms=False, maha=False, adaptive=False, tnms=False),
    f'{P} (full, 8 modules)': lambda fr: OAFTrack(fr),
    'ByteTrack':         lambda fr: ByteTrack(fr),
    'BoT-SORT':          lambda fr: BoTSORT(fr),
    'OC-SORT':           lambda fr: OCSORT(fr),
    'SORT':              lambda fr: SORT(fr),
    'IoU-Tracker':       lambda fr: IoUTracker(fr),
}
ABLATIONS = {
    f'{P} (full)':           lambda fr: OAFTrack(fr),
    'w/o ASG-NMS':           lambda fr: OAFTrack(fr, asg_nms=False),
    'w/o Mahalanobis gate':  lambda fr: OAFTrack(fr, maha=False),
    'w/o Re-ID fusion':      lambda fr: OAFTrack(fr, reid=False),
    'w/o occlusion-adaptive':lambda fr: OAFTrack(fr, adaptive=False),
    'w/o k-reciprocal rec.': lambda fr: OAFTrack(fr, recovery=False),
    'w/o temporal NMS':      lambda fr: OAFTrack(fr, tnms=False),
    'w/o GMC':               lambda fr: OAFTrack(fr, gmc=False),
    'w/o observation output':lambda fr: OAFTrack(fr, obs_out=False),
}
print('Trackers :', list(TRACKERS))
print('Ablations:', list(ABLATIONS))

In [ ]:
# ── Cell 8: Evaluation — HOTA, CLEAR-MOT (MOTA), Identity (IDF1) ────────
# Re-implementation of TrackEval's MotChallenge2DBox metrics (HOTA, CLEAR, Identity).
ALPHAS = np.arange(0.05, 0.99, 0.05)

def load_mot(path, is_gt):
    """Returns {frame: (ids, xyxy)}. GT rows with conf flag 0 (ignore) are dropped."""
    out = {}
    if not Path(path).exists() or Path(path).stat().st_size == 0: return out
    a = np.loadtxt(path, delimiter=',', ndmin=2)
    if a.size == 0: return out
    if is_gt and a.shape[1] >= 7: a = a[a[:, 6] != 0]
    for f in np.unique(a[:, 0]).astype(int):
        r = a[a[:, 0] == f]
        out[f] = (r[:, 1].astype(int), np.c_[r[:, 2], r[:, 3], r[:, 2] + r[:, 4], r[:, 3] + r[:, 5]])
    return out

def eval_sequence(gt_path, res_path):
    gt, tr = load_mot(gt_path, True), load_mot(res_path, False)
    frames = sorted(set(gt) | set(tr))
    gmap = {g: i for i, g in enumerate(sorted({int(x) for f in gt for x in gt[f][0]}))}
    tmap = {t: i for i, t in enumerate(sorted({int(x) for f in tr for x in tr[f][0]}))}
    G, T = len(gmap), len(tmap)
    data = []
    for f in frames:
        gi = np.array([gmap[x] for x in gt[f][0]], int) if f in gt else np.zeros(0, int)
        ti = np.array([tmap[x] for x in tr[f][0]], int) if f in tr else np.zeros(0, int)
        sim = iou_matrix(gt[f][1] if f in gt else np.zeros((0, 4)), tr[f][1] if f in tr else np.zeros((0, 4)))
        data.append((gi, ti, sim))
    r = {'num_gt_dets': sum(len(d[0]) for d in data), 'num_tr_dets': sum(len(d[1]) for d in data),
         'num_gt_ids': G, 'num_tr_ids': T}
    # ---------- HOTA ----------
    A = len(ALPHAS)
    hTP, hFN, hFP, LocA = np.zeros(A), np.zeros(A), np.zeros(A), np.zeros(A)
    pot = np.zeros((G, T)); gcount = np.zeros((G, 1)); tcount = np.zeros((1, T))
    for gi, ti, sim in data:
        if len(gi) and len(ti):
            den = sim.sum(0)[None, :] + sim.sum(1)[:, None] - sim
            s = np.zeros_like(sim); mk = den > np.finfo(float).eps; s[mk] = sim[mk] / den[mk]
            pot[gi[:, None], ti[None, :]] += s
        gcount[gi] += 1; tcount[0, ti] += 1
    gas = pot / np.maximum(gcount + tcount - pot, 1e-12) if G and T else pot
    mcounts = [np.zeros((G, T)) for _ in ALPHAS]
    for gi, ti, sim in data:
        if len(gi) == 0: hFP += len(ti); continue
        if len(ti) == 0: hFN += len(gi); continue
        score = gas[gi[:, None], ti[None, :]] * sim
        mr, mc = linear_sum_assignment(-score)
        for a, al in enumerate(ALPHAS):
            ok = sim[mr, mc] >= al - np.finfo(float).eps
            ar, ac = mr[ok], mc[ok]; n = len(ar)
            hTP[a] += n; hFN[a] += len(gi) - n; hFP[a] += len(ti) - n
            if n: LocA[a] += sim[ar, ac].sum(); mcounts[a][gi[ar], ti[ac]] += 1
    AssA = np.zeros(A)
    for a in range(A):
        mc_ = mcounts[a]
        ass = mc_ / np.maximum(1, gcount + tcount - mc_)
        AssA[a] = (mc_ * ass).sum() / max(1, hTP[a])
    r.update(hTP=hTP, hFN=hFN, hFP=hFP, AssA_x_TP=AssA * hTP, LocA_sum=LocA)
    # ---------- CLEAR MOT ----------
    TP = FN = FP = IDSW = 0; MOTP_sum = 0.0
    prev_id = np.full(G, np.nan); prev_ts = np.full(G, np.nan)
    gid_count = np.zeros(G); gmatched = np.zeros(G); frag = np.zeros(G)
    for gi, ti, sim in data:
        if len(gi) == 0: FP += len(ti); continue
        if len(ti) == 0:
            FN += len(gi); gid_count[gi] += 1; prev_ts[:] = np.nan; continue
        sc = 1000 * (ti[None, :] == prev_ts[gi][:, None]) + sim
        sc[sim < 0.5 - np.finfo(float).eps] = 0
        mr, mc = linear_sum_assignment(-sc)
        ok = sc[mr, mc] > np.finfo(float).eps; mr, mc = mr[ok], mc[ok]
        mg, mt = gi[mr], ti[mc]
        pm = prev_id[mg]
        IDSW += int(np.sum(~np.isnan(pm) & (mt != pm)))
        gid_count[gi] += 1; gmatched[mg] += 1
        not_prev = np.isnan(prev_ts)
        prev_id[mg] = mt; prev_ts[:] = np.nan; prev_ts[mg] = mt
        frag += not_prev & ~np.isnan(prev_ts)
        n = len(mg); TP += n; FN += len(gi) - n; FP += len(ti) - n
        if n: MOTP_sum += sim[mr, mc].sum()
    ratio = gmatched[gid_count > 0] / gid_count[gid_count > 0]
    MT = int(np.sum(ratio > 0.8)); PT = int(np.sum(ratio >= 0.2)) - MT; ML = G - MT - PT
    r.update(TP=TP, FN=FN, FP=FP, IDSW=IDSW, MOTP_sum=MOTP_sum, MT=MT, PT=PT, ML=ML,
             Frag=int(np.sum(np.maximum(0, frag - 1))))
    # ---------- Identity (IDF1) ----------
    pot = np.zeros((G, T)); gc = np.zeros(G); tc = np.zeros(T)
    for gi, ti, sim in data:
        mg, mt = np.nonzero(sim >= 0.5 - np.finfo(float).eps)
        pot[gi[mg], ti[mt]] += 1; gc[gi] += 1; tc[ti] += 1
    if G == 0 or T == 0:
        IDTP, IDFN, IDFP = 0, gc.sum(), tc.sum()
    else:
        fp_mat = np.zeros((G + T, G + T)); fn_mat = np.zeros((G + T, G + T))
        fp_mat[G:, :T] = 1e10; fn_mat[:G, T:] = 1e10
        for g in range(G): fn_mat[g, :T] = gc[g]; fn_mat[g, T + g] = gc[g]
        for t in range(T): fp_mat[:G, t] = tc[t]; fp_mat[t + G, t] = tc[t]
        fn_mat[:G, :T] -= pot; fp_mat[:G, :T] -= pot
        mr, mc = linear_sum_assignment(fn_mat + fp_mat)
        IDFN = fn_mat[mr, mc].sum(); IDFP = fp_mat[mr, mc].sum(); IDTP = gc.sum() - IDFN
    r.update(IDTP=IDTP, IDFN=IDFN, IDFP=IDFP)
    return r

def summarize(per_seq):
    """Combine raw per-sequence counts exactly like TrackEval's COMBINED_SEQ row."""
    S = lambda k: sum(np.asarray(p[k], float) for p in per_seq)
    hTP, hFN, hFP = S('hTP'), S('hFN'), S('hFP')
    DetA = hTP / np.maximum(1, hTP + hFN + hFP)
    AssA = S('AssA_x_TP') / np.maximum(1, hTP)
    HOTA = np.sqrt(DetA * AssA)
    LocA = np.maximum(1e-10, S('LocA_sum')) / np.maximum(1e-10, hTP)
    DetRe = hTP / np.maximum(1, hTP + hFN); DetPr = hTP / np.maximum(1, hTP + hFP)
    TP, FN, FP, IDSW = S('TP'), S('FN'), S('FP'), S('IDSW')
    IDTP, IDFN, IDFP = S('IDTP'), S('IDFN'), S('IDFP')
    pct = lambda x: round(100 * float(x), 3)
    return {
        'HOTA': pct(HOTA.mean()), 'DetA': pct(DetA.mean()), 'AssA': pct(AssA.mean()),
        'DetRe': pct(DetRe.mean()), 'DetPr': pct(DetPr.mean()), 'LocA': pct(LocA.mean()),
        'MOTA': pct((TP - FP - IDSW) / max(1, TP + FN)), 'MOTP': pct(S('MOTP_sum') / max(1, TP)),
        'IDF1': pct(IDTP / max(1, IDTP + 0.5 * IDFP + 0.5 * IDFN)),
        'IDP': pct(IDTP / max(1, IDTP + IDFP)), 'IDR': pct(IDTP / max(1, IDTP + IDFN)),
        'IDSW': int(IDSW), 'FP': int(FP), 'FN': int(FN), 'Frag': int(S('Frag')),
        'MT': int(S('MT')), 'ML': int(S('ML')), 'GT_IDs': int(S('num_gt_ids')), 'Pred_IDs': int(S('num_tr_ids')),
    }

# ---- Sanity checks: GT vs GT must be perfect; an empty result must give 0 ----
s0 = SEQS[0]; gt0 = s0 / 'gt' / 'gt.txt'
chk = summarize([eval_sequence(gt0, gt0)])
print('GT vs GT   :', {k: chk[k] for k in ('HOTA', 'MOTA', 'IDF1', 'IDSW')})
assert chk['MOTA'] == 100 and chk['IDF1'] == 100 and abs(chk['HOTA'] - 100) < 1e-6, 'metric bug'
empty = RESULT_DIR / '_empty.txt'; empty.write_text('')
chk = summarize([eval_sequence(gt0, empty)])
print('Empty file :', {k: chk[k] for k in ('HOTA', 'MOTA', 'IDF1', 'FN')}, ' <- this is what v4 reported')

In [ ]:
# ── Cell 9: Run trackers + evaluate ─────────────────────────────────────
def run_tracker(name, factory, seqs=SEQS, out_root=TRACKER_DIR):
    out_dir = out_root / name.replace('/', '_').replace(' ', '_') / 'data'
    out_dir.mkdir(parents=True, exist_ok=True)
    per_seq, n_rows, t_track, n_frames = [], 0, 0.0, 0
    for s in tqdm(seqs, desc=name, leave=False):
        cache = CACHES[s.name]; trk = factory(read_seqinfo(s)['frameRate'])
        lines = []
        for f in cache.frame_ids:
            b, sc, ft, hs = cache.get(f)
            t0 = time.perf_counter()
            if len(b):
                keep = (nms_asg(b, sc, ft, hs) if trk.frame_filter == 'asg' else nms_standard(b, sc, STD_NMS_IOU))
                b, sc, ft, hs = b[keep], sc[keep], ft[keep], hs[keep]
            outs = trk.update(b, sc, ft, hs, cache.warps[int(f)])
            t_track += time.perf_counter() - t0; n_frames += 1
            for tid, bb, conf in outs:
                lines.append(f'{int(f)},{int(tid)},{bb[0]:.2f},{bb[1]:.2f},{bb[2]-bb[0]:.2f},{bb[3]-bb[1]:.2f},{conf:.4f},-1,-1,-1\n')
        res_file = out_dir / f'{s.name}.txt'
        res_file.write_text(''.join(lines)); n_rows += len(lines)
        r = eval_sequence(s / 'gt' / 'gt.txt', res_file); r['seq'] = s.name
        per_seq.append(r)
    if n_rows == 0:
        raise RuntimeError(f'{name}: tracker wrote 0 boxes – check detections/thresholds')
    m = summarize(per_seq)
    trk_fps = n_frames / max(t_track, 1e-9)
    m.update({'Tracker': name, 'Assoc FPS': round(trk_fps, 1),
              'Total FPS': round(1 / (1 / DET_FPS + (1 / REID_FPS if trk_uses_reid(name) else 0)
                                     + (1 / GMC_FPS if trk_uses_gmc(name) else 0) + 1 / trk_fps), 1)})
    return m, per_seq

def trk_uses_reid(name):
    t = (TRACKERS.get(name) or ABLATIONS.get(name))(25)
    return bool(getattr(t, 'with_reid', False)) or getattr(t, 'frame_filter', '') == 'asg'

def trk_uses_gmc(name):
    t = (TRACKERS.get(name) or ABLATIONS.get(name))(25)
    return bool(getattr(t, 'needs_warp', False))

COLS = ['Tracker', 'HOTA', 'DetA', 'AssA', 'MOTA', 'IDF1', 'MOTP', 'IDSW', 'FP', 'FN', 'Frag', 'MT', 'ML',
        'Assoc FPS', 'Total FPS']
import pickle
CKPT_MAIN = RESULT_DIR / f'checkpoint_main_{TAG}_v6.pkl'   # per detector/Re-ID + code version
if RESUME and CKPT_MAIN.exists():
    main_rows, PER_SEQ = pickle.load(open(CKPT_MAIN, 'rb'))
    print(f'Loaded checkpoint {CKPT_MAIN.name} ({len(main_rows)} trackers) – set RESUME=False to re-run')
else:
    main_rows, PER_SEQ = [], {}
    for name, fac in TRACKERS.items():
        m, ps = run_tracker(name, fac); main_rows.append(m); PER_SEQ[name] = ps
        print(f'{name:14s} HOTA={m["HOTA"]:6.2f}  MOTA={m["MOTA"]:6.2f}  IDF1={m["IDF1"]:6.2f}  IDSW={m["IDSW"]:5d}  '
              f'assoc {m["Assoc FPS"]:.0f} FPS')
    pickle.dump((main_rows, PER_SEQ), open(CKPT_MAIN, 'wb'))
main_df = pd.DataFrame(main_rows)[COLS].sort_values('HOTA', ascending=False).reset_index(drop=True)
main_df.to_csv(RESULT_DIR / 'comparison_table.csv', index=False)
from IPython.display import display
display(main_df)

In [ ]:
# ── Cell 10: Ablation study ─────────────────────────────────────────────
CKPT_AB = RESULT_DIR / f'checkpoint_ablation_{TAG}_v6.pkl'
if RESUME and CKPT_AB.exists():
    ab_rows, ab_seq = pickle.load(open(CKPT_AB, 'rb')); PER_SEQ.update(ab_seq)
    print(f'Loaded checkpoint {CKPT_AB.name} ({len(ab_rows)} variants) – set RESUME=False to re-run')
else:
    ab_rows, ab_seq = [], {}
    for name, fac in ABLATIONS.items():
        m, ps = run_tracker(name, fac, out_root=TRACKER_DIR / '_ablation'); ab_rows.append(m); ab_seq[name] = ps
        print(f'{name:24s} HOTA={m["HOTA"]:6.2f}  MOTA={m["MOTA"]:6.2f}  IDF1={m["IDF1"]:6.2f}  IDSW={m["IDSW"]:5d}')
    pickle.dump((ab_rows, ab_seq), open(CKPT_AB, 'wb')); PER_SEQ.update(ab_seq)
ab_df = pd.DataFrame(ab_rows)[COLS]
full = ab_df.iloc[0]
for k in ('HOTA', 'MOTA', 'IDF1'): ab_df[f'Δ{k}'] = (ab_df[k] - full[k]).round(2)
ab_df.to_csv(RESULT_DIR / 'ablation_table.csv', index=False)
display(ab_df)

In [ ]:
# ── Cell 11: Per-sport breakdown (basketball / football / volleyball) ──
def sport_map():
    # look only in the known locations (a recursive search over all images is extremely slow)
    cands = [VAL_DIR.parent.parent / 'splits_txt', VAL_DIR.parent / 'splits_txt',
             Path(DATA_ROOT) / 'splits_txt', Path(DATA_ROOT) / 'sportsmot_publish' / 'splits_txt']
    m = {}
    for d in [c for c in cands if c.is_dir()][:1]:
        for sp in ('basketball', 'football', 'volleyball'):
            f = d / f'{sp}.txt'
            if f.exists():
                for line in f.read_text().split():
                    m[line.strip()] = sp
    return m
SPORT = sport_map()
if SPORT:
    rows = []
    for name in TRACKERS:
        for sp in ('basketball', 'football', 'volleyball'):
            ps = [r for r in PER_SEQ[name] if SPORT.get(r['seq']) == sp]
            if ps:
                m = summarize(ps); rows.append({'Tracker': name, 'Sport': sp, 'Seqs': len(ps),
                                                **{k: m[k] for k in ('HOTA', 'MOTA', 'IDF1', 'IDSW')}})
    sport_df = pd.DataFrame(rows).pivot(index='Tracker', columns='Sport', values=['HOTA', 'MOTA', 'IDF1'])
    sport_df.to_csv(RESULT_DIR / 'per_sport_table.csv'); display(sport_df.round(2))
else:
    print('splits_txt/ not found under DATA_ROOT – per-sport breakdown skipped')

seq_df = pd.DataFrame([{'Tracker': n, 'Sequence': r['seq'], **{k: v for k, v in summarize([r]).items()
                        if k in ('HOTA', 'MOTA', 'IDF1', 'IDSW', 'FP', 'FN')}}
                       for n in PER_SEQ for r in PER_SEQ[n]])
seq_df.to_csv(RESULT_DIR / 'per_sequence_metrics.csv', index=False)
print('Per-sequence metrics saved:', RESULT_DIR / 'per_sequence_metrics.csv')

In [ ]:
# ── Cell 12: Figures + Markdown summary ─────────────────────────────────
metrics = ['HOTA', 'MOTA', 'IDF1', 'AssA']
colors = ['#2a78d6', '#e8a33d', '#3aa676', '#c9573f']
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(main_df)); w = 0.2
for i, (k, c) in enumerate(zip(metrics, colors)):
    ax.bar(x + (i - 1.5) * w, main_df[k], w, label=k, color=c)
ax.set_xticks(x); ax.set_xticklabels(main_df['Tracker'], rotation=15)
ax.set_ylabel('%'); ax.set_title(f'SportsMOT-{SPLIT}: tracker comparison')
ax.legend(ncol=4, frameon=False); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(RESULT_DIR / 'comparison_chart.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
d = ab_df.iloc[1:]
y = np.arange(len(d)); h = 0.27
for i, (k, c) in enumerate(zip(['ΔHOTA', 'ΔMOTA', 'ΔIDF1'], colors)):
    ax.barh(y + (i - 1) * h, d[k], h, label=k, color=c)
ax.axvline(0, color='k', lw=0.8); ax.set_yticks(y); ax.set_yticklabels(d['Tracker'])
ax.invert_yaxis(); ax.set_xlabel(f'change vs. {PROPOSED_NAME} (full), points')
ax.set_title('Ablation: effect of removing each component'); ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(RESULT_DIR / 'ablation_chart.png', dpi=200); plt.show()

with open(RESULT_DIR / 'summary.md', 'w', encoding='utf-8') as f:
    f.write(f'# {PROPOSED_NAME} — SportsMOT-{SPLIT} results\n\n')
    f.write(f'Detector: {DET_WEIGHTS} @ {IMG_SIZE}px | Re-ID: {REID_BACKBONE} | sequences: {len(SEQS)}\n\n')
    f.write(f'Detector {DET_FPS:.1f} FPS, Re-ID {REID_FPS:.1f} FPS, GMC {GMC_FPS:.1f} FPS\n\n')
    f.write('## 1. Comparison\n\n' + main_df.to_markdown(index=False) + '\n\n')
    f.write('## 2. Ablation\n\n' + ab_df.to_markdown(index=False) + '\n\n')
    if SPORT: f.write('## 3. Per sport\n\n' + sport_df.round(2).to_markdown() + '\n')
print('Saved to', RESULT_DIR)
for p in sorted(RESULT_DIR.iterdir()): print('  ', p.name)

### Cell 13: fine-tune the detector on SportsMOT **train** (step 1 – skip if you downloaded `detector_final.pt`)
A COCO-pretrained YOLO also detects referees and spectators, which SportsMOT does not annotate, so it produces many FPs.
Set `RUN_FINETUNE = True` and run this cell alone (it is self-contained). When training finishes the model is frozen as
`WORK_DIR/detector_final.pt`, which Cell 2 then selects automatically; re-run Cells 1–12 afterwards.
(Only the *train* split is used for training, so the val numbers remain a fair held-out evaluation.)

In [ ]:
# ── Cell 13: Fine-tune the detector on SportsMOT train (SELF-CONTAINED – can run right after a restart) ──
import os, sys, shutil, configparser
from pathlib import Path
import numpy as np, cv2
from tqdm.auto import tqdm
import torch
from ultralytics import YOLO

# ---------------- settings ----------------
DATA_ROOT    = Path(r"data/SportsMOT")
WORK_DIR     = Path(r"outputs/sportsmot")
BASE_MODEL   = 'yolov8s.pt'
EPOCHS       = 30
TRAIN_IMGSZ  = 1280          # use 640 if you must train on CPU
BATCH        = 8             # lower to 4 if "CUDA out of memory"
FRAME_STRIDE = 5             # every 5th train frame (use 10 on CPU)
RESUME_TRAINING = False      # True = continue an interrupted run from last.pt (same device/settings)
RUN_FINETUNE = False         # !! set True ONLY to (re)train the detector – it overwrites best.pt !!

if not RUN_FINETUNE:
    print('Cell 13 skipped (RUN_FINETUNE = False). Your detector is unchanged. For the test submission run Cell 14.')
else:
    # ---------------- device check ----------------
    print('Python :', sys.executable)
    print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
    DEVICE = 0 if torch.cuda.is_available() else 'cpu'
    if DEVICE == 'cpu':
        print('WARNING: training on CPU will be very slow – install the CUDA build of PyTorch first.')
    else:
        print('GPU    :', torch.cuda.get_device_name(0))

    # ---------------- helpers ----------------
    IMG_EXTS = {'.jpg', '.jpeg', '.png'}
    def is_seq(d): return d.is_dir() and (d / 'img1').is_dir()
    def find_split(root, split):
        for c in [root / split, root / 'dataset' / split, root / 'sportsmot_publish' / 'dataset' / split,
                  root / 'dancetrack' / split]:
            if c.is_dir() and any(is_seq(d) for d in c.iterdir()): return c
        raise FileNotFoundError(f'"{split}" split not found under {root}')

    def export(split_dir, part, stride, yroot):
        (yroot / 'images' / part).mkdir(parents=True, exist_ok=True)
        (yroot / 'labels' / part).mkdir(parents=True, exist_ok=True)
        n = 0
        for s in tqdm(sorted(d for d in split_dir.iterdir() if is_seq(d)), desc=f'export {part}'):
            gt_file = s / 'gt' / 'gt.txt'
            if not gt_file.exists(): continue
            gt = np.loadtxt(gt_file, delimiter=',', ndmin=2)
            if gt.shape[1] >= 7: gt = gt[gt[:, 6] != 0]
            cp = configparser.ConfigParser(); cp.read(s / 'seqinfo.ini')
            W, H = int(cp['Sequence']['imWidth']), int(cp['Sequence']['imHeight'])
            frames = sorted(p for p in (s / 'img1').iterdir() if p.suffix.lower() in IMG_EXTS)
            for p in frames[::stride]:
                fno = int(p.stem)
                dst = yroot / 'images' / part / f'{s.name}_{fno:06d}{p.suffix}'
                if not dst.exists(): shutil.copy(p, dst)
                lab = []
                for row in gt[gt[:, 0] == fno]:
                    x, y, w, h = row[2:6]
                    x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
                    if x2 - x1 > 1 and y2 - y1 > 1:
                        cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
                        bw, bh = (x2 - x1) / W, (y2 - y1) / H
                        lab.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
                (yroot / 'labels' / part / f'{dst.stem}.txt').write_text(chr(10).join(lab))
                n += 1
        return n

    # ---------------- dataset (skipped if already exported) ----------------
    yroot = WORK_DIR / 'yolo_sportsmot'
    if not (yroot / 'data.yaml').exists():
        print('train images:', export(find_split(DATA_ROOT, 'train'), 'train', FRAME_STRIDE, yroot))
        print('val images  :', export(find_split(DATA_ROOT, 'val'), 'val', 10 * FRAME_STRIDE, yroot))
        yaml_lines = ['path: ' + yroot.as_posix(),
                      'train: images/train',
                      'val: images/val',
                      'names:',
                      '  0: player']
        (yroot / 'data.yaml').write_text(chr(10).join(yaml_lines) + chr(10))
    else:
        print('Dataset already exported:', yroot)

    # ---------------- train ----------------
    run_dir = WORK_DIR / 'yolo_runs' / 'sportsmot_player'
    last = run_dir / 'weights' / 'last.pt'
    if RESUME_TRAINING and last.exists():
        print('Resuming from', last)
        res = YOLO(str(last)).train(resume=True)
    else:
        res = YOLO(BASE_MODEL).train(
            data=str(yroot / 'data.yaml'), epochs=EPOCHS, imgsz=TRAIN_IMGSZ,
            batch=BATCH, device=DEVICE, project=str(WORK_DIR / 'yolo_runs'),
            name='sportsmot_player', exist_ok=True, workers=0, patience=10)
    best = run_dir / 'weights' / 'best.pt'
    print('\nFine-tuned weights:', best, '| exists:', best.exists())
    # Freeze the finished model as detector_final.pt (read-only, never overwritten)
    import stat
    final_det = WORK_DIR / 'detector_final.pt'
    if last.exists() and not final_det.exists():
        ck = torch.load(last, map_location='cpu', weights_only=False)
        if ck.get('optimizer') is None:                 # optimizer is stripped only when training finished
            shutil.copy2(last, final_det); os.chmod(final_det, stat.S_IREAD)
            print('Saved final detector (read-only):', final_det)
        else:
            print('Training not finished – detector_final.pt NOT created yet.')
    elif final_det.exists():
        print('detector_final.pt already exists – not changed:', final_det)
    print('Cell 2 now picks detector_final.pt automatically – just Run All.')

### Cell 14: SportsMOT **test**-set submission
Run Cells 2–7 first (evaluation cells are not needed). This cell
1. detects the **test** sequences with the same fine-tuned detector (cached, so it only happens once),
2. runs the chosen tracker configuration,
3. writes one `<sequence>.txt` per video in MOTChallenge format and zips them for upload to the official
   SportsMOT server: https://codalab.lisn.upsaclay.fr/competitions/12424 (Participate → Submit).
   Note: that server stopped accepting submissions in 2026, so the paper reports SportsMOT on the validation split.

**Fix the configuration before submitting and submit it once.** Trying many variants on the test server
and reporting the best one is test-set tuning, which reviewers reject.

In [ ]:
# ── Cell 14: SportsMOT TEST-set submission ─────────────────────────────
import zipfile, gc
TEST_TRACKERS = {        # the configuration reported in the paper
    PROPOSED_NAME: TRACKERS[PROPOSED_NAME],
}

TEST_DIR = find_split_dir(DATA_ROOT, 'test')
TEST_SEQS = sorted(d for d in TEST_DIR.iterdir() if _is_seq_dir(d))
print('Test split:', TEST_DIR, '|', len(TEST_SEQS), 'sequences (SportsMOT test has 150)')

TEST_CACHE = WORK_DIR / 'det_cache_test' / TAG
ensure_cache(TEST_SEQS, TEST_CACHE, WORK_DIR / 'det_cache_test' / Path(DET_WEIGHTS).stem, 'test')

SUB_ROOT = WORK_DIR / 'submission' / TAG
for name, fac in TEST_TRACKERS.items():
    safe = name.replace('/', '_').replace(' ', '_')
    out_dir = SUB_ROOT / safe; out_dir.mkdir(parents=True, exist_ok=True)
    n_rows, empty = 0, []
    for s in tqdm(TEST_SEQS, desc=f'track {name}'):
        cache = SeqCache(TEST_CACHE / f'{s.name}.npz')
        trk = fac(read_seqinfo(s)['frameRate'])
        lines = []
        for f in cache.frame_ids:
            b, sc, ft, hs = cache.get(f)
            if len(b):
                if trk.frame_filter == 'asg':
                    keep = nms_asg(b, sc, ft, hs)
                else:
                    keep = nms_standard(b, sc, STD_NMS_IOU)
                b, sc, ft, hs = b[keep], sc[keep], ft[keep], hs[keep]
            for tid, bb, conf in trk.update(b, sc, ft, hs, cache.warps[int(f)]):
                w, h = bb[2] - bb[0], bb[3] - bb[1]
                lines.append(f'{int(f)},{int(tid)},{bb[0]:.2f},{bb[1]:.2f},{w:.2f},{h:.2f},{conf:.4f},-1,-1,-1')
        (out_dir / f'{s.name}.txt').write_text(chr(10).join(lines) + (chr(10) if lines else ''))
        n_rows += len(lines)
        if not lines: empty.append(s.name)
        del cache; gc.collect()
    zip_path = SUB_ROOT / f'{safe}_SportsMOT_test.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in sorted(out_dir.glob('*.txt')):
            z.write(f, arcname=f.name)          # txt files at the root of the zip
    print(f'{name}: {len(TEST_SEQS)} sequence files, {n_rows} boxes -> {zip_path}')
    if empty: print('  WARNING – sequences with no output:', empty[:10])
print('Upload the .zip at https://codalab.lisn.upsaclay.fr/competitions/12424 (Participate -> Submit).')